In [5]:
import os, sys, importlib, pathlib
import pandas as pd
from pathlib import Path
sys.path.append (os.path.abspath(".."))
# my utils
from utils import fa
from utils import modeling
from utils import preprocess_listings

# 标准版存档V0：

LABELS_EN=[
            'open to different cultures', 'cosmopolitan','international view', 'cultural exchange',
            'personal life', 'life experiences', 'divers interests', 'hobbies', 'enjoy life',
            'meet new people', 'welcoming', 'friendly', 'sociable', 'interpersonal interaction',
            'thoughtful service', 'attentive to needs', 'willing to help', 'responsive',
            'fan of Airbnb', 'Airbnb community','love Airbnb', 'travel with Airbnb'
        ]
DICT_FACTOR_ITEMS={"ouverture":['open to different cultures', 'cosmopolitan','international view', 'cultural exchange'],
                   "authenticité":['personal life', 'life experiences', 'divers interests', 'hobbies', 'enjoy life'],
                   "sociabilité":['meet new people', 'welcoming', 'friendly', 'sociable', 'interpersonal interaction'],
                   "auto_promotion":['thoughtful service', 'attentive to needs', 'willing to help', 'responsive'],
                   "exemplarité":['fan of Airbnb', 'Airbnb community','love Airbnb', 'travel with Airbnb']}


In [6]:
path_df="../data_processed\listings_tactics_bio_vis-paris_london-2306_2406.csv"
df_all=pd.read_csv(path_df)
print(df_all.shape)

C:\Users\yeliu\AppData\Local\Temp\ipykernel_8772\3453220627.py:2: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all=pd.read_csv(path_df)


(187851, 124)


In [8]:
df_all.columns.to_list()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

In [10]:
# drop no pic match?
df=df_all.copy()
print(df[['host_has_profile_pic', 'has_face']].value_counts(dropna=False),"\n")
df=df[df["has_face"].notna()]
print(len(df_all),len(df))

host_has_profile_pic  has_face
t                     1.0         123800
                      0.0          55515
f                     NaN           7264
t                     NaN           1271
f                     0.0              1
Name: count, dtype: int64 

187851 179316


In [ ]:
vars_tactics=[
    "ouverture", "authenticité","sociabilité","auto_promotion","exemplarité", 
    "host_picture_type","is_smiling", "smile_score"
    ]
df[vars_tactics].value_counts(dropna=False)
df[vars_tactics]=df[vars_tactics].fillna(0)

# vars_person=["age_class","gender"]
# df[vars_person]=df[vars_person].fillna('unk')



## ols:

In [16]:
df[(df['is_paris']==1)&(df['in_2024']==0)]

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,exemplarité,has_face,bbox_list,host_picture_type,age,age_class,gender,smile_score,is_smiling,dominant_emotion
92212,153674,https://www.airbnb.com/rooms/153674,20230606220143,2023-06-07,city scrape,Rental unit in Paris · ★4.44 · 1 bedroom · 3 b...,"Ideally located in the heart of Paris, in a li...","Staying in the 10th arrondissement, between th...",https://a0.muscache.com/pictures/prohost-api/H...,739021,...,0.010267,1.0,"[[79, 48, 148, 142]]",pro_style,47.0,middle,Woman,99.420952,1.0,happy
92213,5396,https://www.airbnb.com/rooms/5396,20230606220143,2023-06-08,previous scrape,Rental unit in Paris · ★4.55 · Studio · 1 bed ...,"Cozy, well-appointed and graciously designed s...","You are within walking distance to the Louvre,...",https://a0.muscache.com/pictures/52413/f9bf76f...,7903,...,-0.154413,1.0,"[[141, 38, 206, 124], [72, 14, 132, 96]]",life_style,29.0,young,multi,97.692505,1.0,happy
92214,154292,https://www.airbnb.com/rooms/154292,20230606220143,2023-06-07,city scrape,Rental unit in Paris · ★4.62 · 1 bedroom · 1 b...,"Nice flat designed by an architect, in an area...","Charming authentic parisian neighborhood, with...",https://a0.muscache.com/pictures/bfc84997-6491...,137719,...,-0.751471,1.0,"[[88, 62, 152, 146]]",pro_style,26.0,young,Man,0.000939,0.0,neutral
92215,7397,https://www.airbnb.com/rooms/7397,20230606220143,2023-06-08,city scrape,Rental unit in Paris · ★4.72 · 2 bedrooms · 2 ...,"VERY CONVENIENT, WITH THE BEST LOCATION !<br /...",NaN,https://a0.muscache.com/pictures/67928287/330b...,2626,...,-0.709955,1.0,"[[50, 47, 157, 192]]",pro_style,40.0,middle,Man,0.063789,0.0,angry
92216,33814,https://www.airbnb.com/rooms/33814,20230606220143,2023-06-07,city scrape,Rental unit in Paris · ★4.36 · 1 bedroom · 2 b...,"70m2, confortable, clean, quiet, very bright a...","In the neighborhood, you find both big streets...",https://a0.muscache.com/pictures/190719/da5f80...,146066,...,1.229936,0.0,[],no_person,NaN,NaN,NaN,0.000000,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125108,907876483956947091,https://www.airbnb.com/rooms/907876483956947091,20230606220143,2023-06-07,city scrape,Rental unit in Gentilly · ★New · 1 bedroom · 1...,Bel appartement de 30m2. <br />Il se compose d...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,51193419,...,0.000000,1.0,"[[80, 45, 151, 146]]",pro_style,26.0,young,Man,99.821266,1.0,happy
125109,907889552689420357,https://www.airbnb.com/rooms/907889552689420357,20230606220143,2023-06-08,city scrape,Rental unit in Paris · ★New · 1 bedroom · 1 be...,This modern one-bedroom apartment in the 19th ...,The 19th arrondissement in Paris holds down th...,https://a0.muscache.com/pictures/prohost-api/H...,518556594,...,-0.256635,0.0,[],no_person,NaN,NaN,NaN,0.000000,0.0,NaN
125110,907921757183609134,https://www.airbnb.com/rooms/907921757183609134,20230606220143,2023-06-07,city scrape,Rental unit in Paris · ★New · 1 bedroom · 2 be...,Notre mission est de permettre aux individus d...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,489964484,...,0.000000,0.0,[],no_person,NaN,NaN,NaN,0.000000,0.0,NaN
125111,907922174138736701,https://www.airbnb.com/rooms/907922174138736701,20230606220143,2023-06-07,city scrape,Rental unit in Paris · ★New · 1 bedroom · 2 be...,Notre mission est de permettre aux individus d...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,489964484,...,0.000000,0.0,[],no_person,NaN,NaN,NaN,0.000000,0.0,NaN


In [35]:
importlib.reload(modeling)
from utils.modeling import write_formula,build_model

import statsmodels.api as sm
import statsmodels.formula.api as smf


# C(host_is_superhost)
# C(host_has_profile_pic)

x_vars=["host_identity_verified", "host_has_profile_pic",         
    "review_scores_rating", #"has_rating", "number_of_reviews",
    "years_since_host","professional_host",'host_is_superhost', ##'calculated_host_listings_count',
    # 'status_changed', 'old_host_sp_changed',# "is_changed"
    "lang", "len",
    "price",
    "availability_90",#*
    "room_type", "instant_bookable", #'is_within_1km',
    
    "ouverture", "authenticité","sociabilité","auto_promotion","exemplarité", 
    "host_picture_type","is_smiling"
]  
tactics_vars=["is_paris","in_2024"]


formula= ("booking_rate_l90d ~ C(host_identity_verified)  + review_scores_rating + has_rating + "
            "C(host_is_superhost) +"
            "years_since_host + professional_host + "
            "len + C(lang) +"
            "price  + availability_90 + C(room_type) + C(instant_bookable) + "
            "(ouverture + authenticité + sociabilité + auto_promotion + exemplarité)*is_paris+"
            "C(host_picture_type) + is_smiling "
            # smile_score coef 过小!
            # "is_paris "
)
df_ols=df[df['in_2024']==1]
model=smf.ols(formula, data=df_ols).fit()
summary=model.summary()
print(summary)

                            OLS Regression Results                            
Dep. Variable:      booking_rate_l90d   R-squared:                       0.259
Model:                            OLS   Adj. R-squared:                  0.258
Method:                 Least Squares   F-statistic:                     1241.
Date:                Sun, 29 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:00:05   Log-Likelihood:                 41644.
No. Observations:              106812   AIC:                        -8.323e+04
Df Residuals:                  106781   BIC:                        -8.293e+04
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

In [58]:
# subdf=df[df['in_2024']==1]
# print(subdf[['is_paris','in_2024']].value_counts(dropna=False))
# formula= ("booking_rate_l90d ~ C(host_identity_verified)  + C(lang) + C(room_type) + C(instant_bookable) + + review_scores_rating + has_rating + years_since_host + professional_host + len + price + availability_90 + "
#         "C(host_is_superhost) + ouverture + authenticité + sociabilité + auto_promotion + exemplarité +  C(host_picture_type) + is_smiling + "
#         # smile_score coef 过小!
#         "is_paris "
#         # "in_2024"
        
# )
# model=smf.ols(formula, data=subdf).fit()
# summary=model.summary()
# print(summary)

In [68]:
formula= ("is_smiling ~ C(host_identity_verified) + C(host_has_profile_pic) + C(host_is_superhost) + C(lang) + C(room_type) + C(instant_bookable) + review_scores_rating + has_rating + years_since_host + professional_host + len + price + availability_90 + "
        # "ouverture + authenticité + sociabilité + auto_promotion + exemplarité + is_smiling +  C(host_picture_type)+"
        # smile_score coef 过小!
        "is_paris * in_2024"
)
model=smf.ols(formula, data=df).fit()
summary=model.summary()
print(summary)

                            OLS Regression Results                            
Dep. Variable:             is_smiling   R-squared:                       0.060
Model:                            OLS   Adj. R-squared:                  0.060
Method:                 Least Squares   F-statistic:                     569.9
Date:                Mon, 23 Mar 2026   Prob (F-statistic):               0.00
Time:                        22:12:28   Log-Likelihood:            -1.0712e+05
No. Observations:              179316   AIC:                         2.143e+05
Df Residuals:                  179295   BIC:                         2.145e+05
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [6]:
importlib.reload(modeling)
from utils.modeling import modeling_main,build_model

#-------------------------------vars------------------------------------

x_vars=["host_identity_verified", "host_has_profile_pic",         
    "review_scores_rating", #"has_rating", "number_of_reviews",
    "years_since_host","professional_host",'host_is_superhost', ##'calculated_host_listings_count',
    # 'status_changed', 'old_host_sp_changed',# "is_changed"
    "lang", "len",
    "price",
    "availability_90",#*
    "room_type", "instant_bookable", #'is_within_1km',
    
    "ouverture", "authenticité","sociabilité","auto_promotion","exemplarité", 
    "host_picture_type","is_smiling"
]  
tactics_vars=["is_paris","in_2024"]

modeling_main(df_input=df_all, 
            x_vars=x_vars, 
            y_var="booking_rate_l90d", 
            key_vars=tactics_vars, 
            group_col=None,#"host_is_superhost',#==stauts_changed+sp_changed
            output_folder=None,
            to_fillna0=True,
            run_vif=True,
            # save_models_summary=False,  
            save_models_table=False,
            save_plots=False,
            ndigits=4
        )

=============================================check data=============================================
[INFO] fillna in key cols!`

========================================check output folder=========================================
================================================vif=================================================
+key_vars ; - group_cols
remove key_vars in x_vars
Final x_vars_ctrl :['host_identity_verified', 'host_has_profile_pic', 'review_scores_rating', 'years_since_host', 'professional_host', 'host_is_superhost', 'lang', 'len', 'price', 'availability_90', 'room_type', 'instant_bookable', 'ouverture', 'authenticité', 'sociabilité', 'auto_promotion', 'exemplarité', 'host_picture_type', 'is_smiling']

[INFO] formula :
 booking_rate_l90d ~ C(host_identity_verified) + C(host_has_profile_pic) + C(host_is_superhost) + C(lang) + C(room_type) + C(instant_bookable) + C(host_picture_type) + review_scores_rating + years_since_host + professional_host + len + price + availabilit

,Variables,VIF,Niveau_colinearite
0,Intercept,91561.521721,***
1,C(host_identity_verified)[T.t],1.017968,*
2,C(host_has_profile_pic)[T.t],1.000178,*
3,C(host_is_superhost)[T.t],1.095642,*
4,C(lang)[T.fr],1.926281,*
5,C(lang)[T.no_text],1.008223,*
6,C(lang)[T.other_langs],1.038891,*
7,C(room_type)[T.Hotel room],1.042722,*
8,C(room_type)[T.Private room],1.167226,*
9,C(room_type)[T.Shared room],1.007452,*



 ============================================basic model============================================= 

[NUMBER CHECK1] x_vars:19, x_vars_ctrl:19
['host_identity_verified', 'host_has_profile_pic', 'review_scores_rating', 'years_since_host', 'professional_host', 'host_is_superhost', 'lang', 'len', 'price', 'availability_90', 'room_type', 'instant_bookable', 'ouverture', 'authenticité', 'sociabilité', 'auto_promotion', 'exemplarité', 'host_picture_type', 'is_smiling', 'booking_rate_l90d']
-key_vars ; - group_cols
Final x_vars_ctrl :['host_identity_verified', 'host_has_profile_pic', 'review_scores_rating', 'years_since_host', 'professional_host', 'host_is_superhost', 'lang', 'len', 'price', 'availability_90', 'room_type', 'instant_bookable', 'ouverture', 'authenticité', 'sociabilité', 'auto_promotion', 'exemplarité', 'host_picture_type', 'is_smiling']

[INFO] formula :
 booking_rate_l90d ~ C(host_identity_verified) + C(host_has_profile_pic) + C(host_is_superhost) + C(lang) + C(room_type)

TypeError: 'in <string>' requires string as left operand, not NoneType